# W05-03 · 검색 실험과 선택

필수 20분 + 확장 15분. 개발 질의에서 설정을 비교하고 고정된 설정을 별도 확인 질의로 평가합니다.

**진행 방식**: 먼저 실행 결과를 예측하고 셀을 실행합니다. 변경 실습은 제공된 값 하나를 바꿉니다. 핵심 셀은 네트워크·API 키 없이 실행됩니다. 모든 정책·질의는 교육용 합성 데이터입니다.

VS Code → 우측 상단 Select Kernel → 이 저장소의 .venv (Python 3.11). 노트북별로 Restart 후 Run All 가능합니다.

In [ ]:
from pathlib import Path
import sys, json, math, statistics
from dataclasses import asdict, replace
from collections import Counter
from time import perf_counter

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course_labs").is_dir()), None)
assert ROOT is not None, "저장소 내부에서 notebook을 열어 주세요."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_labs.retrieval import (
    Document,
    Chunk,
    LexicalIndex,
    load_documents,
    load_cases,
    chunk_document,
    tokenize,
    expand_query,
    eligible_chunks,
    reciprocal_rank_fusion,
    rerank_by_coverage,
    retrieval_metrics,
    retrieve,
    answer_from_evidence,
    evaluate,
)
from IPython.display import display, HTML

docs = load_documents()
cases = load_cases()
print("Python:", sys.version.split()[0], "| 문서:", len(docs), "| 평가 질의:", len(cases))
assert sys.version_info[:2] == (3, 11), "과정 기준 Python 3.11.x 커널 선택"

In [ ]:
def show(rows, columns=None):
    import html

    if not rows:
        return display(HTML("<p>결과 없음</p>"))
    cols = columns or list(rows[0])
    headers = "".join(
        '<th style="padding:8px;text-align:left">' + html.escape(str(c)) + "</th>" for c in cols
    )
    body = "".join(
        "<tr>"
        + "".join(
            '<td style="padding:8px;border-bottom:1px solid #ddd">'
            + html.escape(str(r.get(c, "")))
            + "</td>"
            for c in cols
        )
        + "</tr>"
        for r in rows
    )
    display(
        HTML(
            '<table style="font-size:15px;border-collapse:collapse"><tr>'
            + headers
            + "</tr>"
            + body
            + "</table>"
        )
    )

## 01 기준 설정

실험 전에 비교 기준과 성공 조건을 적습니다.

In [ ]:
dev = [c for c in cases if c["split"] == "dev"]
test = [c for c in cases if c["split"] == "test"]
baseline_config = {
    "method": "bm25",
    "size": 120,
    "overlap": 24,
    "k": 3,
    "expand": False,
    "rerank": False,
}
baseline = evaluate(dev, **baseline_config)
print(baseline["summary"])

**관찰·변경 과제**

성공 조건은 평균 Recall 증가 + 접근 제한 유지 + 특정 실패 질의 개선처럼 구체적으로 쓰세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 02 하나의 변경

동의어 확장만 비교합니다.

In [ ]:
expanded = evaluate(dev, **{**baseline_config, "expand": True})
before = {r["id"]: r for r in baseline["rows"]}
show(
    [
        {
            "id": r["id"],
            "before": before[r["id"]]["retrieved"],
            "after": r["retrieved"],
            "recall_before": before[r["id"]]["recall"],
            "recall_after": r["recall"],
        }
        for r in expanded["rows"]
    ]
)

**관찰·변경 과제**

개선한 Q04 외에 악화한 질의가 있는지 확인하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 03 k 변경

관련 문서 회수와 불필요한 문서 증가를 같이 봅니다.

In [ ]:
show([{"k": k, **evaluate(dev, **{**baseline_config, "k": k})["summary"]} for k in [1, 3, 5]])

**관찰·변경 과제**

컨텍스트에 추가한 문서 수가 생성 비용과 근거 혼란에 미치는 영향을 기록하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 04 설정 탐색

작은 탐색 공간을 개발 데이터에서만 비교합니다.

In [ ]:
candidates = []
for method in ["bm25", "tfidf", "rrf"]:
    for expand in [False, True]:
        cfg = {**baseline_config, "method": method, "expand": expand}
        e = evaluate(dev, **cfg)
        candidates.append({"config": cfg, "result": e})
show(
    [
        {"method": c["config"]["method"], "expand": c["config"]["expand"], **c["result"]["summary"]}
        for c in candidates
    ]
)

**관찰·변경 과제**

여기서는 6개 조합만 탐색합니다. 확인 데이터를 탐색에 넣지 않습니다.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 05 실패 유형

검색 실패와 답변 보류 실패를 구분합니다.

In [ ]:
for row in expanded["rows"]:
    if row["recall"] is not None and row["recall"] < 1:
        print("검색 누락:", row["id"], row["query"], row["retrieved"])
    if not row["relevant"] and not row["abstained"]:
        print("잘못된 발췌:", row["id"])

**관찰·변경 과제**

정답 문서 누락, 잘못된 날짜, 권한 누출, 부족한 근거 각각 담당 모듈을 지정하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 06 시간 측정

외부 모델 없이 작은 자료에서 측정한 로컬 시간입니다.

In [ ]:
timings = []
for _ in range(20):
    timings.append(retrieve("반품 배송비", **baseline_config)["elapsed_ms"])
ordered = sorted(timings)
print(
    "median ms:",
    statistics.median(timings),
    "p95 nearest-rank ms:",
    ordered[math.ceil(0.95 * len(ordered)) - 1],
)

**관찰·변경 과제**

20회 작은 측정값을 프로덕션 p95 보장으로 주장하지 않습니다.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 07 재현 식별 정보

자료와 설정이 같아야 결과를 비교할 수 있습니다.

In [ ]:
r = retrieve("반품 배송비", **baseline_config)
print("corpus_hash:", r["corpus_hash"])
print(json.dumps(r["config"], ensure_ascii=False, sort_keys=True))

**관찰·변경 과제**

검색기의 코드 버전과 토큰화 규칙도 기록하려면 어떤 필드를 추가하나요?

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 08 설정 선택

점수가 같은 후보에서는 단순한 후보를 우선 검토합니다.

In [ ]:
selected_config = {**baseline_config, "expand": True}
print("선택:", selected_config)
print("이유: Q04 표현 차이 대응. 동의어 오확장은 추가 확인 필요.")

**관찰·변경 과제**

본인의 선택 이유를 평균 점수·실패 질의·운영 부담 세 항목으로 설명하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 09 별도 확인

선택이 끝난 뒤 확인 세트를 사용합니다.

In [ ]:
confirmed = evaluate(test, **selected_config)
print(confirmed["summary"])
show(confirmed["rows"], ["id", "relevant", "retrieved", "recall", "mrr", "abstained"])

**관찰·변경 과제**

Q14처럼 날짜 숫자 때문에 검색이 흔들리는 경우 다음 새 확인 질의로 추가하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 10 보고서 제출

실패 사례 하나가 설계 개선 요구사항이 됩니다.

In [ ]:
final_report = {
    "baseline": baseline,
    "selected_config": selected_config,
    "dev_selected": expanded,
    "test": confirmed,
    "decision": "추가 확인 후 적용",
    "next_check": "날짜 표현과 복합 질문 개선",
}
print(json.dumps(final_report, ensure_ascii=False, indent=2)[:800])

**관찰·변경 과제**

최종 실험 결과를 8주차 PRD의 평가 계획 및 위험 항목과 연결하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 완료 체크

- 설정 하나를 바꾼 비교 결과와 실패 질의 1개
- 출처·문서 버전·기준 날짜를 포함한 설명
- 측정한 것과 측정하지 않은 것의 구분

## 참고

- [RAG 원 논문](https://arxiv.org/abs/2005.11401)
- [Stanford IR: TF-IDF](https://nlp.stanford.edu/IR-book/html/htmledition/term-frequency-and-weighting-1.html)
- [Stanford IR: BM25](https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html)
- [Ollama embed API](https://docs.ollama.com/api/embed)

로컬 구현 위치: course_labs/retrieval.py. 형태소 분석·학습된 임베딩·신경망 재정렬을 구현한 것으로 해석하지 않습니다.